<a href="https://colab.research.google.com/github/MuhammadAyyanHassan/flyrank-ml-internship-work/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAyyanHassan/flyrank-ml-internship-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# Setup: connect to the FlyRank warehouse

%pip -q install duckdb scikit-learn

import os
import duckdb
import pandas as pd
import numpy as np

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add your Read token to Colab Secrets as HF_TOKEN."
    )

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(
    "CREATE OR REPLACE SECRET hf_token "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

MARCH_REL = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

APRIL_REL = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-04/*.parquet"
)

print("Warehouse connection configured.")
print("Decision window: March 2026")
print("Future outcome window: April 2026")

Warehouse connection configured.
Decision window: March 2026
Future outcome window: April 2026


In [2]:
# Verify that the March partition is accessible

schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{MARCH_REL}')"
).df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [3]:
# Build one March row per content-client pair.
# April is NOT used here.

march_sql = f"""
WITH march_daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,

        gsc_data_available,
        ga4_data_available,

        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,

        ga4_pageviews,
        ga4_sessions,
        ga4_engaged_sessions,

        sessions_organic,
        sessions_direct,
        sessions_referral,
        sessions_social,
        sessions_paid,
        sessions_ai,

        scroll_events

    FROM read_parquet('{MARCH_REL}')
    WHERE month = '2026-03'
),

march_frame AS (
    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(*) AS observed_days,

        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,

        AVG(gsc_avg_position) AS march_avg_position,

        SUM(ga4_pageviews) AS march_pageviews,
        SUM(ga4_sessions) AS march_sessions,
        SUM(ga4_engaged_sessions) AS march_engaged_sessions,

        SUM(sessions_organic) AS march_organic_sessions,
        SUM(sessions_ai) AS march_ai_sessions,

        SUM(scroll_events) AS march_scroll_events,

        BOOL_OR(gsc_data_available) AS has_gsc_data,
        BOOL_OR(ga4_data_available) AS has_ga4_data

    FROM march_daily
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM march_frame
"""

march_frame = con.execute(march_sql).df()

print("March decision rows:", len(march_frame))
display(march_frame.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March decision rows: 331437


,client_hash_id,content_hash_id,observed_days,march_impressions,march_clicks,march_avg_position,march_pageviews,march_sessions,march_engaged_sessions,march_organic_sessions,march_ai_sessions,march_scroll_events,has_gsc_data,has_ga4_data
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,31,1140.0,2.0,4.394234,0.0,0.0,0.0,0.0,0.0,0.0,True,False
1,client_73cda7b4e4f265ea,content_05597932fe4da067,31,57.0,0.0,2.714744,0.0,0.0,0.0,0.0,0.0,0.0,True,False
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,31,149.0,0.0,6.481453,4.0,4.0,0.0,0.0,0.0,0.0,True,True
3,client_73cda7b4e4f265ea,content_05434271b257bb68,31,1421.0,6.0,6.320337,12.0,9.0,0.0,4.0,0.0,1.0,True,True
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,31,2770.0,16.0,4.459107,3.0,3.0,0.0,0.0,0.0,0.0,True,True
5,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,31,48.0,0.0,14.753175,0.0,0.0,0.0,0.0,0.0,0.0,True,False
6,client_73cda7b4e4f265ea,content_2662845f598544ef,31,150.0,1.0,6.341880,1.0,1.0,0.0,0.0,0.0,0.0,True,True
7,client_73cda7b4e4f265ea,content_22610b0934f8825e,31,67.0,0.0,12.791667,0.0,0.0,0.0,0.0,0.0,0.0,True,False
8,client_73cda7b4e4f265ea,content_712c365258cee05c,31,6048.0,23.0,4.950311,8.0,8.0,0.0,6.0,0.0,0.0,True,True
9,client_73cda7b4e4f265ea,content_476c37c366920c1b,31,223.0,0.0,50.390299,1.0,1.0,0.0,0.0,0.0,0.0,True,True


In [4]:
# Cell 4 — Inspect March signal distributions

signals = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_pageviews",
    "march_sessions",
    "march_organic_sessions",
    "march_ai_sessions",
    "march_scroll_events",
]

signal_summary = (
    march_frame[signals]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .T
    .round(2)
)

display(signal_summary)

,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
march_impressions,331437.0,846.79,4044.51,0.0,0.0,2.00,216.00,1707.00,4225.0,14909.28,617124.0
march_clicks,331437.0,2.48,19.65,0.0,0.0,0.00,0.00,3.00,11.0,47.00,5668.0
march_avg_position,176738.0,16.00,17.69,0.0,5.0,8.51,20.37,40.07,56.5,79.76,309.0
march_pageviews,260737.0,5.69,31.82,0.0,0.0,0.00,1.00,8.00,24.0,115.00,2879.0
march_sessions,260737.0,4.99,28.52,0.0,0.0,0.00,1.00,6.00,21.0,101.00,2730.0
march_organic_sessions,260737.0,2.25,17.48,0.0,0.0,0.00,0.00,3.00,9.0,46.00,3171.0
march_ai_sessions,260737.0,0.03,0.45,0.0,0.0,0.00,0.00,0.00,0.0,1.00,74.0
march_scroll_events,260737.0,0.84,5.47,0.0,0.0,0.00,0.00,1.00,4.0,17.00,667.0


In [5]:
# Cell 5 — CTR and position relationship

march_frame["march_ctr"] = np.where(
    march_frame["march_impressions"] > 0,
    march_frame["march_clicks"] / march_frame["march_impressions"],
    np.nan
)

ctr_position = march_frame[
    [
        "march_impressions",
        "march_clicks",
        "march_ctr",
        "march_avg_position",
    ]
].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).T.round(4)

display(ctr_position)

print("Rows with impressions > 0:",
      int((march_frame["march_impressions"] > 0).sum()))

print("Rows with clicks > 0:",
      int((march_frame["march_clicks"] > 0).sum()))

print("Rows with valid CTR:",
      int(march_frame["march_ctr"].notna().sum()))

,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
march_impressions,331437.0,846.7902,4044.5148,0.0,0.000,2.0000,216.0000,1707.0000,4225.0000,14909.2800,617124.0
march_clicks,331437.0,2.4796,19.6513,0.0,0.000,0.0000,0.0000,3.0000,11.0000,47.0000,5668.0
march_ctr,176738.0,0.0046,0.0378,0.0,0.000,0.0000,0.0022,0.0062,0.0109,0.0588,1.0
march_avg_position,176738.0,15.9993,17.6863,0.0,5.002,8.5053,20.3692,40.0669,56.5000,79.7592,309.0


Rows with impressions > 0: 176738
Rows with clicks > 0: 68837
Rows with valid CTR: 176738


In [ ]:
# Cell 6 — Build April outcome data for evaluation only
# IMPORTANT: April is NOT used to create the baseline score.

april_sql = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS april_impressions,
    SUM(gsc_clicks) AS april_clicks,
    AVG(gsc_avg_position) AS april_avg_position,
    SUM(ga4_pageviews) AS april_pageviews,
    SUM(ga4_sessions) AS april_sessions,
    SUM(sessions_organic) AS april_organic_sessions,
    SUM(sessions_ai) AS april_ai_sessions
FROM read_parquet('{APRIL_REL}')
WHERE month = '2026-04'
GROUP BY
    client_hash_id,
    content_hash_id
"""

april_frame = con.execute(april_sql).df()

print("April outcome rows:", len(april_frame))
display(april_frame.head(10))

In [ ]:
# Cell 7 — Join March decisions with April outcomes
# April columns are evaluation-only.

evaluation_frame = march_frame.merge(
    april_frame,
    on=["client_hash_id", "content_hash_id"],
    how="left",
    validate="one_to_one"
)

print("Evaluation rows:", len(evaluation_frame))
print(
    "Rows with April outcomes:",
    int(evaluation_frame["april_impressions"].notna().sum())
)

display(
    evaluation_frame[
        [
            "client_hash_id",
            "content_hash_id",
            "march_impressions",
            "march_clicks",
            "march_avg_position",
            "april_impressions",
            "april_clicks",
            "april_avg_position",
        ]
    ].head(10)
)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.